# SA-Jev · 环境与数据准备
先跑 pilot。正式训练数据约 8,000 HotpotQA + 4,000 MuSiQue；每题最多 3 个状态。所有模型输入排除答案、分解和 gold 标签。

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'pilot'  # Change to 'full' only after the pilot completes and memory is measured.
# MODE = 'full'  
ROOT = DRIVE / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
print('Code:', PROJECT, 'Artifacts:', ROOT)


In [ ]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config()
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    subprocess.run([sys.executable, '-m', 'sa_jev.cli', command, '--root', str(ROOT), *map(str,args)], check=True)


In [ ]:
# Credentials are read from Colab Secrets or environment; never printed or saved.
if 'google.colab' in sys.modules:
    from google.colab import userdata
    for key in ('HF_TOKEN', 'OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY'):
        if not os.environ.get(key):
            try:
                value = userdata.get(key)
                if value: os.environ[key] = value
            except Exception:
                pass


In [ ]:
cli('smoke', '--device', 'cuda' if __import__('torch').cuda.is_available() else 'cpu')

下载原始数据：pilot 每个文件 64 条；full 读取完整源。2Wiki 的官方 dev.json 另置于 ROOT/raw/2wiki_dev.json。MuSiQue 两种官方格式均受支持。

In [ ]:
from sa_jev.preparation import download_sources, download_2wiki, normalize_sources
download_sources(ROOT/'raw', limit=64 if MODE=='pilot' else None)
if MODE=='full':
    download_2wiki(ROOT/'raw')
questions, manifest = normalize_sources(ROOT/'raw', ROOT/'data')
manifest

固定 LightRAG 混合召回。原始段落统一入库，不索引答案或标签。此步骤会调用所配置的抽取与嵌入服务；先检查 configs/research.yaml。首次建图通常比训练 pilot 更耗时。

In [ ]:
from sa_jev.experiments import cache_pools
await cache_pools(ROOT, config)

In [ ]:
from sa_jev.preparation import prepare_tasks
prepare_tasks(ROOT/'data', counts=config['data']['train_counts'], mine=True, dense=config['data']['dense_mining'])

训练前完整扫描公开 OpenJevData。检出的重叠保留记录并在论文报告；未检出不代表证明完全未见。此步骤不使用独立扰动测试集调参。

In [ ]:
cli('overlap')
cli('review-packets')

In [ ]:
from sa_jev.io import read_json
read_json(ROOT/'data/task_manifest.json')

核对实际 K 分布和有效标签数量。缺候选不复制补齐，未知相关性不硬判负例。若排序有效状态太少，核查候选注释映射/补充核验，不能把掩码简单删掉。把独立扰动草稿人工核验后保存为 perturbations_reviewed.jsonl。